# Chapter 5: Model Evaluation and Improvement

## Overview
This notebook covers techniques for reliably evaluating and tuning machine learning models:
1. **Cross-Validation**: $k$-Fold, Stratified $k$-Fold, Leave-One-Out, and GroupKFold validation strategies.
2. **Grid Search**: Systematic hyperparameter tuning using `GridSearchCV`.
3. **Evaluation Metrics for Classification**: Confusion matrices, Precision, Recall, $F_1$-Score, Precision-Recall curves, and ROC-AUC.
4. **Multi-class Evaluation**: Aggregation strategies (micro, macro, and weighted averages).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold,
    GridSearchCV,
    KFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.svm import SVC

# Matplotlib global parameters
plt.rc("font", size=10)
plt.rc("axes", labelsize=11, titlesize=12)

## 1. Cross-Validation

Rather than relying on a single train-test split, **Cross-Validation (CV)** splits data into $k$ equal-sized folds. The model is trained on $k-1$ folds and evaluated on the remaining fold, repeating this process $k$ times:

- **Standard $k$-Fold**: Randomly splits data into $k$ parts. Suitable for regression or balanced datasets.
- **Stratified $k$-Fold**: Preserves the class proportions in each fold. Default for classification in Scikit-Learn.
- **GroupKFold**: Ensures that samples belonging to the same group (e.g., patient IDs, subject recordings) do not appear in both train and validation splits.

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

logreg = LogisticRegression(max_iter=10000)

# 1. Standard Stratified 5-Fold CV (Default for classification)
scores_strat = cross_val_score(logreg, X, y, cv=5)
print(f"Stratified 5-Fold CV Scores: {scores_strat}")
print(f"Mean Accuracy: {scores_strat.mean()*100:.2f}% (± {scores_strat.std()*100:.2f}%)\n")

# 2. Unstratified 5-Fold CV
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(logreg, X, y, cv=kfold)
print(f"Standard 5-Fold CV Scores:   {scores_kfold}")
print(f"Mean Accuracy: {scores_kfold.mean()*100:.2f}%\n")

# 3. GroupKFold (Simulating grouped patient data)
np.random.seed(42)
groups = np.random.randint(0, 10, size=len(y))  # 10 patient groups
group_kfold = GroupKFold(n_splits=5)
scores_group = cross_val_score(logreg, X, y, cv=group_kfold, groups=groups)
print(f"Group 5-Fold CV Scores:      {scores_group}")
print(f"Mean Accuracy: {scores_group.mean()*100:.2f}%")

## 2. Hyperparameter Tuning with Grid Search

`GridSearchCV` exhaustively tests specified hyperparameter combinations using internal cross-validation.

> **Important**: To evaluate tuned models objectively, keep a separate **hold-out test set** untouched until the full Grid Search cross-validation procedure completes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Parameter grid for RBF Support Vector Machine
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "gamma": [0.0001, 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    estimator=SVC(kernel="rbf"),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_*100:.2f}%")
print(f"Hold-out Test Set Accuracy:    {grid_search.score(X_test, y_test)*100:.2f}%")

# Visualize hyperparameter grid cross-validation performance
results = pd.DataFrame(grid_search.cv_results_)
scores_matrix = results.pivot(index="param_C", columns="param_gamma", values="mean_test_score")

plt.figure(figsize=(7, 5))
plt.imshow(scores_matrix, cmap="viridis")
plt.colorbar(label="Mean Cross-Validation Score")
plt.xticks(range(len(param_grid["gamma"])), param_grid["gamma"])
plt.yticks(range(len(param_grid["C"])), param_grid["C"])
plt.xlabel("gamma")
plt.ylabel("C")
plt.title("GridSearch Cross-Validation Accuracy Heatmap")

for i in range(len(param_grid["C"])):
    for j in range(len(param_grid["gamma"])):
        plt.text(j, i, f"{scores_matrix.iloc[i, j]:.2f}", ha="center", va="center", color="w" if scores_matrix.iloc[i, j] < 0.92 else "b")

plt.tight_layout()
plt.show()

## 3. Classification Metrics Beyond Accuracy

Accuracy is often misleading on imbalanced datasets. Better evaluation requires analyzing specific decision trade-offs:

- **Precision**: $\frac{\text{TP}}{\text{TP} + \text{FP}}$ (Out of all samples predicted as positive, how many were correct?)
- **Recall (Sensitivity)**: $\frac{\text{TP}}{\text{TP} + \text{FN}}$ (Out of all actual positive samples, how many were identified?)
- **$F_1$-Score**: Harmonic mean of Precision and Recall: $\frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$
- **ROC-AUC**: Area under the Receiver Operating Characteristic curve plotting True Positive Rate against False Positive Rate across all thresholds.

In [ ]:
# Create an imbalanced binary dataset (90% Class 0, 10% Class 1)
digits = load_digits()
y_imbalanced = (digits.target == 9).astype(int)  # Classify digit '9' vs rest

X_tr_imb, X_te_imb, y_tr_imb, y_te_imb = train_test_split(
    digits.data, y_imbalanced, test_size=0.25, random_state=42, stratify=y_imbalanced
)

svm_imb = SVC(gamma="scale").fit(X_tr_imb, y_tr_imb)
y_pred_imb = svm_imb.predict(X_te_imb)

print("Confusion Matrix:")
print(confusion_matrix(y_te_imb, y_pred_imb))

print("\nDetailed Classification Report:")
print(classification_report(y_te_imb, y_pred_imb, target_names=["Not-9", "Digit-9"]))

# Plot ROC and Precision-Recall Curves
decision_scores = svm_imb.decision_function(X_te_imb)
auc_score = roc_auc_score(y_te_imb, decision_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC Curve
RocCurveDisplay.from_predictions(y_te_imb, decision_scores, ax=axes[0], color="teal", name="SVM")
axes[0].plot([0, 1], [0, 1], "k--", label="Random Classifier (AUC = 0.5)")
axes[0].set_title(f"ROC Curve (AUC = {auc_score:.4f})")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_te_imb, decision_scores)
axes[1].plot(recall, precision, color="crimson", lw=2)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Multi-class Evaluation Metrics

Evaluating multi-class problems involves aggregating class-specific metrics:
- **Macro Averaging**: Computes metrics per class and takes an unweighted average. Weighs every class equally, making it effective for highlighting poor performance on minority classes.
- **Weighted Averaging**: Computes metrics per class and weights them by class support (instance counts).
- **Micro Averaging**: Computes total True Positives, False Negatives, and False Positives globally across all classes.

In [ ]:
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=42, stratify=digits.target
)

rf_multi = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_m, y_tr_m)
y_pred_m = rf_multi.predict(X_te_m)

print("Multi-Class Classification Performance on Digits Dataset:")
print(classification_report(y_te_m, y_pred_m))

f1_macro = f1_score(y_te_m, y_pred_m, average="macro")
f1_weighted = f1_score(y_te_m, y_pred_m, average="weighted")

print(f"Macro-averaged F1 Score:    {f1_macro:.4f}")
print(f"Weighted-averaged F1 Score: {f1_weighted:.4f}")